# Break-My-VLA · M3 Auto-Chaos

在 Colab 选择 L4、G4 或 A100，然后点 **Run all**。本地 RTX 5090 可在 Linux 或 WSL 的 M3 仓库中运行同一个 Notebook。GPU 环境使用 Python 3.11、CUDA 12.8 和无窗口的 MuJoCo 渲染。

Pi0.5 默认使用 PaliGemma tokenizer。先在 Hugging Face 接受模型访问条款，将有权访问的 `HF_TOKEN` 加到 Colab Secrets 并为该 Notebook 启用访问。本地可沿用已登录的 Hugging Face 凭据。[PaliGemma tokenizer 仓库](https://huggingface.co/google/paligemma-3b-pt-224)

Colab 将结果写入 `MyDrive/Break-My-VLA/results/m3_auto_chaos/<运行时间>/`。本地结果写入仓库的 `benchmarks/results/m3_auto_chaos/`。每次运行生成 `metrics.json`、`gallery.json` 和对照 GIF。

## 1. 选择评测范围

默认运行两种模型、一个任务，以及等级 0 和 3。将 `LEVELS` 改为 `"0-5"` 可覆盖完整矩阵。A100 上会记录 FP8 为不可用。可设置 `BMVLA_MOCK=1`，在 CPU 上检查相同的产物流程。

In [ ]:
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

MOCK = os.environ.get("BMVLA_MOCK") == "1"
MODELS = os.environ.get("BMVLA_MODELS", "smolvla,pi05")
PRECISIONS = "bf16,fp8,int8"
TASKS = "libero_object:0"
LEVELS = os.environ.get("BMVLA_LEVELS", "0,3")
MAX_STEPS = int(os.environ.get("BMVLA_MAX_STEPS", "40" if MOCK else "280"))
SEED = 0

## 2. 定位源码和结果目录

优先使用当前 M3 checkout，也可通过 `BMVLA_REPO_DIR` 指定位置。Colab 可自动拉取 M3 分支，也可从 Drive 的 `Break-My-VLA/Break-My-VLA-m3.zip` 解压现有源码包。

In [ ]:
try:
    from google.colab import drive, userdata
except ImportError:
    drive = userdata = None

drive_dir = None
if drive is not None and not MOCK:
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/Break-My-VLA")
    models = {name.strip() for name in MODELS.split(",")}
    if "pi05" in models and not os.environ.get("HF_TOKEN"):
        try:
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            print("Pi0.5 will use any Hugging Face credentials already available in the runtime.")

marker = Path("scripts/m3_gpu_runtime.py")
if repo_dir := os.environ.get("BMVLA_REPO_DIR"):
    repo = Path(repo_dir).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    repo = next((path for path in (cwd, *cwd.parents) if (path / marker).is_file()), None)
    if repo is None and drive is not None:
        repo = Path("/content/Break-My-VLA-m3")
        archive = drive_dir / "Break-My-VLA-m3.zip" if drive_dir else None
        if not repo.exists():
            if archive is not None and archive.is_file():
                with ZipFile(archive) as bundle:
                    bundle.extractall(repo)
            else:
                subprocess.run(
                    [
                        "git", "clone", "--depth", "1", "--branch", "feat/m3-multi-vla",
                        "https://github.com/JasperXzy/Break-My-VLA.git", str(repo),
                    ],
                    check=True,
                )
if repo is None or not (repo / marker).is_file():
    raise FileNotFoundError("Set BMVLA_REPO_DIR to a checkout of feat/m3-multi-vla")

output_root = (
    drive_dir / "results/m3_auto_chaos"
    if drive_dir else repo / "benchmarks/results/m3_auto_chaos"
)
stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
output = Path(os.environ.get("BMVLA_OUTPUT_DIR", str(output_root / stamp))).expanduser().resolve()
print(f"Results will be saved in {output}")

## 3. 运行评测

脚本用 `uv` 创建独立的 Python 3.11 环境，并启用 `--index-strategy unsafe-best-match` 和 `MPLBACKEND=Agg`。Pi0.5 使用自己的 Transformers 环境。评测依次加载模型，释放前一个策略进程的 GPU 显存后再启动下一个。

In [ ]:
command = [sys.executable, "-m", "scripts.m3_gpu_runtime", "--models", MODELS]
if MOCK:
    command.append("--mock")
elif drive is not None:
    command.extend(("--runtime-dir", "/content/bmvla-m3-runtime", "--install-osmesa"))
command.extend((
    "--", "--precisions", PRECISIONS, "--tasks", TASKS, "--levels", LEVELS,
    "--max-steps", str(MAX_STEPS), "--seed", str(SEED), "--output", str(output),
))
subprocess.run(command, cwd=repo, check=True)

## 4. 查看榜单与对照 GIF

榜单的延迟来自 Clean 策略调用。各轴分数按成功率统计，原始回合和逐次调用记录保存在 `metrics.json`。更多参数和计分方式见 [Auto-Chaos 说明](../benchmarks/auto_chaos.md)。

In [ ]:
import json

from IPython.display import Image, Markdown, display

gallery = json.loads((output / "gallery.json").read_text(encoding="utf-8"))
table = [
    "| Model | Precision | Score | Clean p50 (ms) | Peak VRAM (GiB) |",
    "| --- | --- | ---: | ---: | ---: |",
]
for entry in gallery["entries"]:
    table.append(
        f"| {entry['model']} | {entry['precision'].upper()} | {entry['overall_score']:.1%} "
        f"| {entry['latency']['p50_ms']:.1f} | {entry['vram_gb']:.2f} |"
    )
display(Markdown("\n".join(table)))
if gallery["replays"]:
    replay = gallery["replays"][0]
    display(Markdown(f"**{replay['title']}**"))
    for label in ("baseline", "degraded"):
        display(Markdown(replay[f"{label}_label"]))
        display(Image(filename=str(output / replay[f"{label}_gif"]), width=320))